In [8]:
import os
import numpy as np
import pandas as pd

DATA_FILE = "08_food_delivery_updated-1.csv"
OUT_DIR = "outputs"
FAST_LIMIT, STANDARD_LIMIT = 30, 45

def load_data(path):
    df = pd.read_csv(path)
    print("Shape:", df.shape, "Columns:", df.columns.tolist())
    print(df.dtypes)
    print(df.head())
    return df

def clean_data(df):
    print(df.isnull().sum())
    missing = int(df["Delivery_Time_Min"].isnull().sum())
    city_median = df.groupby("City")["Delivery_Time_Min"].transform("median")
    df["Delivery_Time_Min"] = df["Delivery_Time_Min"].fillna(city_median)
    print("Filled missing delivery times:", missing)
    print("Duplicate Order_ID:", int(df["Order_ID"].duplicated().sum()))
    return df.drop_duplicates(subset="Order_ID")

def status_summary(df):
    table = pd.DataFrame({"Orders": df["Order_Status"].value_counts(),
                          "Percent": (df["Order_Status"].value_counts(normalize=True) * 100).round(2)})
    print("Total orders:", len(df))
    print(table)
    return table

def city_performance(df):
    g = df.groupby("City")
    table = pd.DataFrame({
        "Total_Orders": g.size(),
        "Avg_Delivery_Time": g["Delivery_Time_Min"].mean().round(2),
        "Cancellation_Rate": g["Order_Status"].apply(lambda s: (s == "Cancelled").mean() * 100).round(2),
        "Delay_Rate": g["Order_Status"].apply(lambda s: (s == "Delayed").mean() * 100).round(2),
    })
    table = table.sort_values("Avg_Delivery_Time")
    print(table)
    return table

def restaurant_ranking(df):
    g = df.groupby("Restaurant")
    table = pd.DataFrame({
        "Total_Orders": g.size(),
        "Total_Order_Value": g["Order_Value"].sum().round(2),
        "Avg_Order_Value": g["Order_Value"].mean().round(2),
    })
    table = table.sort_values("Total_Order_Value", ascending=False)
    print(table.head(5))
    return table

def peak_hour_summary(df):
    table = df.groupby("Order_Hour").size().sort_values(ascending=False).rename("Orders").to_frame()
    print(table.head(5))
    return table

def status_delivery_time(df):
    table = df.groupby("Order_Status")["Delivery_Time_Min"].agg(["mean", "median", "min", "max"]).round(2)
    print(table)
    return table

def numpy_statistics(df):
    times = df["Delivery_Time_Min"].to_numpy()
    q1, q3 = np.percentile(times, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print("Median:", np.median(times), "Std:", round(np.std(times), 2))
    print("Q1:", q1, "Q3:", q3, "IQR:", iqr, "Lower:", lower, "Upper:", upper)
    return lower, upper

def long_deliveries(df, lower, upper):
    flagged = df[(df["Delivery_Time_Min"] < lower) | (df["Delivery_Time_Min"] > upper)].sort_values("Delivery_Time_Min", ascending=False)
    print("Unusually long deliveries:", len(flagged))
    print(flagged)
    return flagged

def add_bands(df):
    conditions = [df["Delivery_Time_Min"] < FAST_LIMIT, df["Delivery_Time_Min"] <= STANDARD_LIMIT]
    df["Delivery_Band"] = np.select(conditions, ["Fast", "Standard"], default="Delayed")
    print(df["Delivery_Band"].value_counts())
    print((df["Delivery_Band"].value_counts(normalize=True) * 100).round(2))
    return df

def delay_analysis(df):
    table = (pd.crosstab(df["City"], df["Delivery_Band"], normalize="index") * 100).round(2)
    print(table)
    return table

def revenue_tables(df):
    pivot = df.pivot_table(index="City", columns="Payment_Mode", values="Order_Value", aggfunc="sum", margins=True).round(2)
    print(pivot)
    return pivot

def build_recommendations(df, city, restaurant, peak, status):
    slowest = city["Avg_Delivery_Time"].idxmax()
    worst_cancel, worst_delay = city["Cancellation_Rate"].idxmax(), city["Delay_Rate"].idxmax()
    top_restaurant, low_aov = restaurant["Total_Order_Value"].idxmax(), restaurant["Avg_Order_Value"].idxmin()
    top_hours = peak.head(3).index.tolist()
    share = (df.groupby("Payment_Mode")["Order_Value"].sum() / df["Order_Value"].sum() * 100).round(1)
    top_pay = share.idxmax()
    return [
        f"Add rider capacity in {slowest}, which has the longest average delivery time ({city.loc[slowest, 'Avg_Delivery_Time']} min).",
        f"Investigate cancellations in {worst_cancel} ({city.loc[worst_cancel, 'Cancellation_Rate']}%) and delays in {worst_delay} ({city.loc[worst_delay, 'Delay_Rate']}%); overall {status.loc['Cancelled', 'Percent']}% of orders are cancelled and {status.loc['Delayed', 'Percent']}% are delayed.",
        f"Schedule extra riders around the busiest hours {top_hours}, where order volume is highest.",
        f"Give {top_restaurant}, the top revenue restaurant, priority dispatch and run offers to lift the average order value of {low_aov}.",
        f"{top_pay} payments bring the largest share of order value ({share[top_pay]}%), so keep that payment channel reliable and consider offers on the weaker payment modes.",
    ]

def save_outputs(tables, recommendations, df):
    os.makedirs(OUT_DIR, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(os.path.join(OUT_DIR, name + ".csv"))
    with open(os.path.join(OUT_DIR, "recommendations.txt"), "w") as f:
        for i, text in enumerate(recommendations, 1):
            f.write(f"{i}. {text}\n")
            print(i, text)
    df.to_csv(os.path.join(OUT_DIR, "cleaned_food_delivery.csv"), index=False)

def main():
    df = clean_data(load_data(DATA_FILE))
    status = status_summary(df)
    city = city_performance(df)
    restaurant = restaurant_ranking(df)
    peak = peak_hour_summary(df)
    by_status = status_delivery_time(df)
    lower, upper = numpy_statistics(df)
    flagged = long_deliveries(df, lower, upper)
    df = add_bands(df)
    delay = delay_analysis(df)
    revenue = revenue_tables(df)
    recommendations = build_recommendations(df, city, restaurant, peak, status)
    tables = {"status_summary": status, "city_performance": city, "restaurant_ranking": restaurant,
              "peak_hour_summary": peak, "status_delivery_time": by_status, "long_deliveries": flagged,
              "delay_analysis": delay, "revenue_city_payment": revenue}
    save_outputs(tables, recommendations, df)

if __name__ == "__main__":
    main()


Shape: (600, 9) Columns: ['Order_ID', 'City', 'Restaurant', 'Order_Value', 'Distance_km', 'Delivery_Time_Min', 'Order_Status', 'Payment_Mode', 'Order_Hour']
Order_ID              object
City                  object
Restaurant            object
Order_Value          float64
Distance_km          float64
Delivery_Time_Min    float64
Order_Status          object
Payment_Mode          object
Order_Hour             int64
dtype: object
  Order_ID       City   Restaurant  Order_Value  Distance_km  \
0  FD00000      Noida   Green Bowl       588.00          2.6   
1  FD00001      Delhi  Pizza Point       496.68          5.3   
2  FD00002  Faridabad  Pizza Point       419.41          6.0   
3  FD00003      Noida  Pizza Point       195.48          5.0   
4  FD00004  Faridabad      Cafe 24       621.70          2.5   

   Delivery_Time_Min Order_Status Payment_Mode  Order_Hour  
0               32.0    Delivered         Card          10  
1               36.0    Delivered         Cash          22  
